# Day 2 in pandas: the SQL queries, rewritten

Each section below is the pandas equivalent of one `.sql` file from Day 2. Every section ends with a `compare(...)` call that runs the matching SQL file and checks that both tools return the same numbers. If a SQL file doesn't exist yet, the check is skipped.

Run the cells from top to bottom.

In [4]:
import pandas as pd
import sqlite3
import os


os.makedirs("data/raw", exist_ok=True)
# 1. Read the Excel file into a DataFrame
df = pd.read_excel("data/raw/default of credit card clients.xls", header=1)

# 2. Tidy the column names (lowercase, no spaces)
df.columns = df.columns.str.lower().str.replace(" ", "_")



os.makedirs("data/processed", exist_ok=True)
conn = sqlite3.connect("data/processed/credit.db")

# 3. Open (or create) the SQLite database file
conn = sqlite3.connect("data/processed/credit.db")

# 4. Write the DataFrame into the database as a table
df.to_sql("clients", conn, if_exists="replace", index=False)

conn.close()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/default of credit card clients.xls'

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "processed" / "credit.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH}. Rerun the Day 1 loading step."

conn = sqlite3.connect(DB_PATH)
raw = pd.read_sql("SELECT * FROM clients", conn)
print(raw.shape)

# Recreate the clean view so the SQL files can be run for comparison
view_file = ROOT / "sql" / "00_clean_view.sql"
if view_file.exists():
    conn.executescript(view_file.read_text())

AssertionError: Database not found at c:\Users\user\Downloads\data\processed\credit.db. Rerun the Day 1 loading step.

## Helpers
`compare()` checks pandas against SQL. `rate_by()` is the "count customers and default rate per group" pattern that most queries repeat. `ntile()` mimics SQL's `NTILE`.

In [ ]:
def run_sql(filename):
    return pd.read_sql((ROOT / "sql" / filename).read_text(), conn)


def compare(filename, pandas_result, sort_by=None):
    """Run the matching .sql file and check it agrees with the pandas result."""
    path = ROOT / "sql" / filename
    if not path.exists():
        print(f"[skipped]  {filename} not found")
        return
    a = run_sql(filename).copy()
    b = pandas_result.copy()
    for frame in (a, b):
        for c in frame.columns:
            if not pd.api.types.is_numeric_dtype(frame[c]):
                frame[c] = frame[c].astype(str)
    if sort_by:
        a = a.sort_values(sort_by)
        b = b.sort_values(sort_by)
    a = a.reset_index(drop=True)
    try:
        b = b[list(a.columns)].reset_index(drop=True)
        pd.testing.assert_frame_equal(a, b, check_dtype=False, check_exact=False, rtol=1e-6)
        print(f"[match]    {filename}")
    except (AssertionError, KeyError) as e:
        print(f"[MISMATCH] {filename}\n{e}")


def rate_by(frame, col):
    """Customers and default rate (%) for each value of `col`."""
    g = frame.groupby(col, observed=True)["is_default"].agg(customers="size", default_rate_pct="mean")
    g["default_rate_pct"] = (100 * g["default_rate_pct"]).round(2)
    return g.reset_index()


def ntile(n_rows, k):
    """Bucket numbers 1..k for n_rows sorted rows, like SQL NTILE(k)."""
    sizes = [n_rows // k + (1 if i < n_rows % k else 0) for i in range(k)]
    return np.repeat(np.arange(1, k + 1), sizes)

## 00: Clean view
Equivalent of `00_clean_view.sql`: readable column names, labelled categories, age bands.

- `CASE WHEN ... END` becomes `.map()` (with `.fillna()` for the ELSE branch) or `pd.cut()` for numeric ranges.
- `AS` aliases become `.rename()`.

In [ ]:
MONTHS = ["sep", "aug", "jul", "jun", "may", "apr"]   # most recent month first
PAY_OLD  = ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]
BILL_OLD = [f"bill_amt{i}" for i in range(1, 7)]
PAID_OLD = [f"pay_amt{i}" for i in range(1, 7)]

rename = {"default_payment_next_month": "is_default"}
rename.update({old: f"pay_{m}" for old, m in zip(PAY_OLD, MONTHS)})
rename.update({old: f"bill_{m}" for old, m in zip(BILL_OLD, MONTHS)})
rename.update({old: f"paid_{m}" for old, m in zip(PAID_OLD, MONTHS)})

df = raw.rename(columns=rename).copy()

df["sex"] = df["sex"].map({1: "Male", 2: "Female"})
df["education"] = (
    df["education"]
    .map({1: "Graduate school", 2: "University", 3: "High school"})
    .fillna("Others/Unknown")
)
df["marital_status"] = df["marriage"].map({1: "Married", 2: "Single"}).fillna("Others/Unknown")
df["age_band"] = pd.cut(df["age"],
                        bins=[0, 25, 35, 45, 55, np.inf], right=False,
                        labels=["<25", "25-34", "35-44", "45-54", "55+"]
)

PAY_COLS = [f"pay_{m}" for m in MONTHS]   # Sep, Aug, Jul, Jun, May, Apr

df.shape

## 01: Portfolio overview
`COUNT(*)`, `SUM()`, `AVG()` become `len()`, `.sum()`, `.mean()`.

In [ ]:
q01 = pd.DataFrame({
    "customers": [len(df)],
    "defaults": [int(df["is_default"].sum())],
    "default_rate_pct": [round(100 * df["is_default"].mean(), 2)],
    "avg_limit": [round(df["limit_bal"].mean())],
})
compare("01_portfolio_overview.sql", q01)
q01

## 02: Default rate by segment
`GROUP BY` becomes `.groupby()`. `UNION ALL` becomes `pd.concat()`. The scalar subquery `(SELECT SUM(is_default) ...)` is just a variable computed first.

In [ ]:
total_defaults = df["is_default"].sum()

def segment_summary(col):
    g = df.groupby(col, observed=True)["is_default"].agg(customers="size", n_def="sum", rate="mean")
    return pd.DataFrame({
        "dimension": col,
        "segment": g.index.astype(str),
        "customers": g["customers"].values,
        "default_rate_pct": (100 * g["rate"]).round(2).values,
        "share_of_defaults_pct": (100 * g["n_def"] / total_defaults).round(2).values,
    })

q02 = pd.concat([segment_summary(c) for c in ["education", "marital_status", "age_band", "sex"]],
                ignore_index=True)
q02 = q02.sort_values(["dimension", "default_rate_pct"], ascending=[True, False]).reset_index(drop=True)
compare("02_default_by_segment.sql", q02, sort_by=["dimension", "segment"])
q02

## 03: Credit limit deciles
`NTILE(10) OVER (ORDER BY limit_bal)` becomes: sort, then assign bucket numbers with the `ntile()` helper.

Many customers share the same limit, so the order *within* a tie is arbitrary. To make SQL and pandas agree exactly, add `id` as a tie-breaker in the SQL file: `NTILE(10) OVER (ORDER BY limit_bal, id)`.

In [ ]:
d = df.sort_values(["limit_bal", "id"]).copy()
d["limit_decile"] = ntile(len(d), 10)

q03 = (d.groupby("limit_decile")
         .agg(min_limit=("limit_bal", "min"),
              max_limit=("limit_bal", "max"),
              customers=("is_default", "size"),
              default_rate_pct=("is_default", "mean"))
         .reset_index())
q03["default_rate_pct"] = (100 * q03["default_rate_pct"]).round(2)
compare("03_limit_deciles.sql", q03)
q03

## 04: Utilization bands
A `CASE WHEN` with ranges becomes `np.select(conditions, labels, default=...)`. Conditions are checked in order, like in SQL.

In [ ]:
utilization = df["bill_sep"] / df["limit_bal"]

df["utilization_band"] = np.select(
    [utilization <= 0, utilization < 0.3, utilization < 0.6, utilization < 0.9],
    ["0. No balance", "1. Under 30%", "2. 30-59%", "3. 60-89%"],
    default="4. 90%+",
)

q04 = rate_by(df, "utilization_band")
compare("04_utilization_bands.sql", q04)
q04

## 05: Delinquency history
In SQL, `(pay_sep >= 1) + (pay_aug >= 1) + ...` adds booleans. In pandas, compare all six columns at once and sum across each row with `axis=1`.

In [ ]:
df["months_late"] = (df[PAY_COLS] >= 1).sum(axis=1)

q05 = rate_by(df, "months_late")
compare("05_delinquency_history.sql", q05)
q05

In [ ]:
# The second version suggested in the guide: group by the latest status alone
q05b = rate_by(df, "pay_sep")
q05b

## 06: Roll-rate matrix
The SQL window `SUM(COUNT(*)) OVER (PARTITION BY from_state)` becomes `groupby(...).transform("sum")`, which puts each group's total on every row of that group.

In [ ]:
def delinquency_state(s):
    return np.select(
        [s <= 0, s == 1, s == 2],
        ["0_Current", "1_One month late", "2_Two months late"],
        default="3_Three+ months late",
    )

states = pd.DataFrame({
    "from_state": delinquency_state(df["pay_aug"]),
    "to_state": delinquency_state(df["pay_sep"]),
})

q06 = states.groupby(["from_state", "to_state"]).size().rename("customers").reset_index()
q06["pct_of_from_state"] = (100 * q06["customers"] / q06.groupby("from_state")["customers"].transform("sum")).round(2)
compare("06_roll_rates.sql", q06, sort_by=["from_state", "to_state"])
q06

In [ ]:
# The same result as a matrix (rows = from, columns = to), ready for Excel
roll_matrix = (pd.crosstab(states["from_state"], states["to_state"], normalize="index") * 100).round(2)
roll_matrix

In [ ]:
# Stretch goal: stack every consecutive month pair (Apr->May, May->Jun, ... Aug->Sep)
pairs = []
for earlier, later in zip(PAY_COLS[1:], PAY_COLS[:-1]):      # e.g. pay_aug -> pay_sep
    pairs.append(pd.DataFrame({
        "from_state": delinquency_state(df[earlier]),
        "to_state": delinquency_state(df[later]),
    }))
all_pairs = pd.concat(pairs, ignore_index=True)

roll_all = all_pairs.groupby(["from_state", "to_state"]).size().rename("customers").reset_index()
roll_all["pct_of_from_state"] = (100 * roll_all["customers"] / roll_all.groupby("from_state")["customers"].transform("sum")).round(2)
roll_all

## 07: Payment ratio
September's payment goes against the **August** statement, so the ratio is `paid_sep / bill_aug`. `.where(bill > 0)` turns non-positive bills into `NaN`, which mirrors the SQL `NULL` and avoids divide-by-zero.

In [ ]:
pay_ratio = df["paid_sep"] / df["bill_aug"].where(df["bill_aug"] > 0)

df["payment_band"] = np.select(
    [pay_ratio.isna(), pay_ratio < 0.05, pay_ratio < 0.25, pay_ratio < 1],
    ["0. No balance due", "1. Under 5% (minimum or less)", "2. 5-24%", "3. 25-99%"],
    default="4. Paid in full",
)

q07 = rate_by(df, "payment_band")
compare("07_payment_ratio.sql", q07)
q07

## 08: Rule-based risk tiers
Same `np.select` pattern, with `|` for OR. The scalar subqueries become plain variables. `lift` is the tier's default rate divided by the portfolio's.

In [ ]:
util_sep = df["bill_sep"] / df["limit_bal"]

df["risk_tier"] = np.select(
    [df["pay_sep"] >= 2, (df["pay_sep"] == 1) | (util_sep >= 0.9)],
    ["High", "Medium"],
    default="Low",
)

portfolio_rate = df["is_default"].mean()
total_defaults = df["is_default"].sum()

t = df.groupby("risk_tier")["is_default"].agg(customers="size", defaults="sum", rate="mean")
q08 = pd.DataFrame({
    "risk_tier": t.index,
    "customers": t["customers"].values,
    "pct_of_customers": (100 * t["customers"] / len(df)).round(2).values,
    "default_rate_pct": (100 * t["rate"]).round(2).values,
    "pct_of_defaults": (100 * t["defaults"] / total_defaults).round(2).values,
    "lift": (t["rate"] / portfolio_rate).round(2).values,
}).sort_values("default_rate_pct", ascending=False).reset_index(drop=True)

compare("08_risk_tiers.sql", q08)
q08

## 09: Rank segment combinations
`HAVING COUNT(*) >= 200` becomes a boolean filter after the aggregation. `RANK()` becomes `.rank(method="min")`, which gives tied segments the same rank, as SQL does.

In [ ]:
q09 = (df.groupby(["age_band", "education"], observed=True)["is_default"]
         .agg(customers="size", defaults="sum", default_rate_pct="mean")
         .reset_index())
q09["default_rate_pct"] = (100 * q09["default_rate_pct"]).round(2)

q09 = q09[q09["customers"] >= 200].copy()
q09["risk_rank"] = q09["default_rate_pct"].rank(method="min", ascending=False).astype(int)
q09 = q09.sort_values("risk_rank").reset_index(drop=True)

compare("09_segment_ranking.sql", q09, sort_by=["age_band", "education"])
q09

## Export for Excel (Day 4)
Writes the same CSVs the Day 2 guide described, so you can use them whichever route you took.

In [ ]:
out = ROOT / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)

exports = {
    "q02_segments.csv": q02,
    "q03_limit_deciles.csv": q03,
    "q04_utilization.csv": q04,
    "q05_delinquency.csv": q05,
    "q06_roll_rates.csv": q06,
    "q06_roll_matrix.csv": roll_matrix.reset_index(),
    "q07_payment_ratio.csv": q07,
    "q08_risk_tiers.csv": q08,
    "q09_segment_ranking.csv": q09,
}
for name, frame in exports.items():
    frame.to_csv(out / name, index=False)
print(f"Saved {len(exports)} files to {out}")